<a href="https://colab.research.google.com/github/Matheus-Santos-AI/CardioIA_FIAP/blob/main/DADOS_CARDIO_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.5 MB/s eta 0:00:00


In [3]:
import kagglehub

path = kagglehub.dataset_download("m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.72G/1.72G [00:26<00:00, 68.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset/versions/1


In [5]:
!pip install pandas

In [6]:
import pandas as pd
import os

base_path = "/root/.cache/kagglehub/datasets/m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset/versions/1/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

csv_path = os.path.join(base_path, "ptbxl_database.csv")
df = pd.read_csv(csv_path, index_col='ecg_id')



print(df.shape)



(21799, 27)


In [7]:
print(df.columns)

Index(['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site',
       'device', 'recording_date', 'report', 'scp_codes', 'heart_axis',
       'infarction_stadium1', 'infarction_stadium2', 'validated_by',
       'second_opinion', 'initial_autogenerated_report', 'validated_by_human',
       'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems',
       'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr'],
      dtype='object')


##Gerando Dataset e Imagens de pacientes que sofreram infarto

In [34]:
df_limpo = df.dropna(subset =['patient_id', 'age', 'sex', 'height', 'weight', 'report', 'scp_codes', 'filename_lr', 'filename_hr'] )
print(df_limpo.iloc[0])

patient_id                                          9086.0
age                                                   48.0
sex                                                      1
height                                               172.0
weight                                                72.0
nurse                                                  0.0
site                                                   0.0
device                                           CS-12   E
recording_date                         1985-12-11 11:50:40
report                               sinusrytm normalt ekg
scp_codes                       {'NORM': 100.0, 'SR': 0.0}
heart_axis                                             NaN
infarction_stadium1                                    NaN
infarction_stadium2                                    NaN
validated_by                                           NaN
second_opinion                                       False
initial_autogenerated_report                         Fal

In [9]:
import pandas as pd
import ast
import os

# carregar os dois arquivos
df = pd.read_csv(os.path.join(base_path, "ptbxl_database.csv"), index_col='ecg_id')
scp_df = pd.read_csv(os.path.join(base_path, "scp_statements.csv"), index_col=0)

# converter scp_codes de string para dicionário de verdade
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

In [10]:
def traduzir_codigos(scp_codes_dict):
    """Recebe {'NORM': 80.0} e devolve as descrições correspondentes"""
    resultados = []
    for codigo, confianca in scp_codes_dict.items():
        if codigo in scp_df.index:
            descricao = scp_df.loc[codigo, 'description']
            classe = scp_df.loc[codigo, 'diagnostic_class']
            resultados.append(f"{codigo} ({descricao}, classe={classe}, confiança={confianca}%)")
    return resultados

# testar em um paciente
print(traduzir_codigos(df.loc[2, 'scp_codes']))

['NORM (normal ECG, classe=NORM, confiança=80.0%)', 'SBRAD (sinus bradycardia, classe=nan, confiança=0.0%)']


In [11]:
def pegar_superclasses(scp_codes_dict):
    classes = set()
    for codigo in scp_codes_dict.keys():
        if codigo in scp_df.index:
            classe = scp_df.loc[codigo, 'diagnostic_class']
            if pd.notna(classe):
                classes.add(classe)
    return list(classes)

df['diagnostic_superclass'] = df['scp_codes'].apply(pegar_superclasses)

print(df[['scp_codes', 'diagnostic_superclass']].head())

                                       scp_codes diagnostic_superclass
ecg_id                                                                
1       {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}                [NORM]
2                   {'NORM': 80.0, 'SBRAD': 0.0}                [NORM]
3                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
4                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
5                     {'NORM': 100.0, 'SR': 0.0}                [NORM]


In [37]:
# todos os pacientes com infarto
filtro_mi = df[df['diagnostic_superclass'].apply(lambda x: 'MI' in x)]
print(filtro_mi.iloc[0])
print(filtro_mi.shape)

patient_id                                                                11275.0
age                                                                          48.0
sex                                                                             0
height                                                                        NaN
weight                                                                       95.0
nurse                                                                         2.0
site                                                                          0.0
device                                                                  CS-12   E
recording_date                                                1984-12-01 14:49:52
report                          sinusrhythmus linkstyp qrs(t) abnormal    infe...
scp_codes                                  {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
heart_axis                                                                    LAD
infarction_stadi

In [38]:
df_limpo = filtro_mi.dropna(subset =['patient_id', 'age', 'sex', 'height', 'weight', 'report', 'scp_codes', 'filename_lr', 'filename_hr'] )
print(df_limpo.iloc[0])
print(df_limpo.shape)

patient_id                                                                13447.0
age                                                                          45.0
sex                                                                             0
height                                                                      182.0
weight                                                                       90.0
nurse                                                                         NaN
site                                                                          3.0
device                                                                  CS-12   E
recording_date                                                1986-02-23 11:41:43
report                          sinusrytm ospecifikt skÄnkelblock avvikande qr...
scp_codes                       {'IMI': 35.0, 'ISCLA': 100.0, 'SEHYP': 50.0, '...
heart_axis                                                                    NaN
infarction_stadi

In [115]:
#Cria o dataframe somente com as colunas relevantes para o modelo
df_final_infarto = pd.DataFrame(df_limpo[['patient_id','age', 'sex', 'report', 'infarction_stadium1','filename_lr', 'filename_hr','diagnostic_superclass' ]])
df_final_infarto['IMC'] = df_limpo['weight']/((df_limpo['height']/100)**2)  #Calcula IMC do paciente
#df_final['sex'] = df_final['sex'].apply(lambda x: 'Masculino' if x == 1 else 'Feminino')
df_final_infarto = df_final_infarto.drop_duplicates(subset='patient_id', keep='first')

print(df_final_infarto.shape)
print(df_final_infarto.iloc[0])


(1278, 9)
patient_id                                                         13447.0
age                                                                   45.0
sex                                                                      0
report                   sinusrytm ospecifikt skÄnkelblock avvikande qr...
infarction_stadium1                                                unknown
filename_lr                                      records100/00000/00146_lr
filename_hr                                      records500/00000/00146_hr
diagnostic_superclass                                      [MI, STTC, HYP]
IMC                                                              27.170632
Name: 146, dtype: object


In [116]:

df_final_infarto['diagnostic_superclass'] = df_final_infarto['diagnostic_superclass'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

df_final_infarto = df_final_infarto.dropna(subset=['diagnostic_superclass'])


df_balanceado_inf = df_final_infarto.groupby('diagnostic_superclass', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 50), random_state=42)
)

print(df_balanceado_inf.iloc[0])
print(df_balanceado_inf.shape)

patient_id                                                          4098.0
age                                                                  300.0
sex                                                                      0
report                   premature ventricular contraction(s). probable...
infarction_stadium1                                                unknown
filename_lr                                      records100/02000/02019_lr
filename_hr                                      records500/02000/02019_hr
diagnostic_superclass                                                   MI
IMC                                                               17.96875
Name: 2019, dtype: object
(50, 9)


/tmp/ipykernel_7862/621200746.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanceado_inf = df_final_infarto.groupby('diagnostic_superclass', group_keys=False).apply(


In [117]:
import wfdb
import matplotlib.pyplot as plt
import os

os.makedirs("temp_images_inf", exist_ok=True)

for ecg_id in df_balanceado_inf.index:
    # pega a linha do paciente no df principal (que tem o filename_lr)
    paciente = df.loc[ecg_id]
    nome_arquivo = f"temp_images_inf/{paciente['patient_id']}.png"

    # checa se já existe ANTES de processar, pra economizar tempo
    if os.path.exists(nome_arquivo):
        print(f"Já existe, pulando: {nome_arquivo}")
        continue

    record_path = os.path.join(base_path, paciente['filename_lr'])

    # lê o sinal
    record = wfdb.rdrecord(record_path)

    # plota e salva
    fig = wfdb.plot_wfdb(record=record, title=f'ECG - paciente {ecg_id}', return_fig=True)
    fig.savefig(nome_arquivo, dpi=150)
    plt.close(fig)  # fecha a figura pra não acumular memória no loop

    print(f"Salvo: {nome_arquivo}")

print("Concluído!")

Salvo: temp_images_inf/4098.0.png
Salvo: temp_images_inf/5431.0.png
Salvo: temp_images_inf/1922.0.png
Salvo: temp_images_inf/3088.0.png
Salvo: temp_images_inf/5136.0.png
Salvo: temp_images_inf/6446.0.png
Salvo: temp_images_inf/6175.0.png
Salvo: temp_images_inf/13485.0.png
Salvo: temp_images_inf/305.0.png
Salvo: temp_images_inf/6585.0.png
Salvo: temp_images_inf/6097.0.png
Salvo: temp_images_inf/2195.0.png
Salvo: temp_images_inf/6053.0.png
Salvo: temp_images_inf/570.0.png
Salvo: temp_images_inf/4930.0.png
Salvo: temp_images_inf/381.0.png
Salvo: temp_images_inf/1429.0.png
Salvo: temp_images_inf/816.0.png
Salvo: temp_images_inf/1342.0.png
Salvo: temp_images_inf/316.0.png
Salvo: temp_images_inf/3897.0.png
Salvo: temp_images_inf/1026.0.png
Salvo: temp_images_inf/3570.0.png
Salvo: temp_images_inf/546.0.png
Salvo: temp_images_inf/1008.0.png
Salvo: temp_images_inf/5231.0.png
Salvo: temp_images_inf/4920.0.png
Salvo: temp_images_inf/5421.0.png
Salvo: temp_images_inf/1147.0.png
Salvo: temp_images_

In [133]:
import os

pasta = "temp_images_inf"
qtd = len(os.listdir(pasta))
print(f"Total de arquivos: {qtd}")

Total de arquivos: 50


In [120]:
import shutil
from google.colab import files

# Compacta a pasta "temp_images_inf" em um arquivo zip
shutil.make_archive("temp_images_inf", 'zip', "temp_images_inf")

# Gera do arquivo zip
files.download("temp_images_inf.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Gerando dataset de pessoas sem a condição infarto

In [125]:
#Filtra todos os dados que o pacience não infartou
filtro_not_mi = df[df['diagnostic_superclass'].apply(lambda x: not 'MI' in x)]
print(filtro_not_mi.iloc[0])
print(filtro_not_mi.shape)

patient_id                                                       15709.0
age                                                                 56.0
sex                                                                    1
height                                                               NaN
weight                                                              63.0
nurse                                                                2.0
site                                                                 0.0
device                                                         CS-12   E
recording_date                                       1984-11-09 09:17:34
report                            sinusrhythmus periphere niederspannung
scp_codes                       {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
heart_axis                                                           NaN
infarction_stadium1                                                  NaN
infarction_stadium2                                

In [126]:
#Remove linhas que possuam dados faltantes nas colunas selecionadas
df_limpo_sec = filtro_not_mi.dropna(subset =['patient_id', 'age', 'sex', 'height', 'weight', 'report', 'scp_codes', 'filename_lr', 'filename_hr'] )
print(df_limpo_sec.iloc[250])
print(df_limpo_sec.shape)

patient_id                                          639.0
age                                                  78.0
sex                                                     1
height                                              155.0
weight                                               46.0
nurse                                                 1.0
site                                                  1.0
device                                         AT-6     6
recording_date                        1987-05-09 13:47:40
report                              trace only requested.
scp_codes                                 {'NST_': 100.0}
heart_axis                                            NaN
infarction_stadium1                                   NaN
infarction_stadium2                                   NaN
validated_by                                          0.0
second_opinion                                      False
initial_autogenerated_report                        False
validated_by_h

In [127]:
#Cria o dataframe somente com as colunas relevantes para o modelo
df_final_norm = pd.DataFrame(df_limpo_sec[['patient_id','age', 'sex', 'report', 'infarction_stadium1','filename_lr', 'filename_hr','diagnostic_superclass' ]])
df_final_norm['IMC'] = df_limpo_sec['weight']/((df_limpo_sec['height']/100)**2) #Calcula IMC do paciente
#df_final_norm['sex'] = df_final_norm['sex'].apply(lambda x: 'Masculino' if x == 1 else 'Feminino')
df_final_norm = df_final_norm.drop_duplicates(subset='patient_id', keep='first')  #removendo dados duplicados

print(df_final_norm.shape)
print(df_final_norm.iloc[252])


(5191, 9)
patient_id                                  1062.0
age                                           77.0
sex                                              1
report                   sinus rhythm. normal ecg.
infarction_stadium1                            NaN
filename_lr              records100/00000/00835_lr
filename_hr              records500/00000/00835_hr
diagnostic_superclass                       [NORM]
IMC                                      20.936639
Name: 835, dtype: object


In [130]:
#Seleciona 50 amostras de cada tipo de diagnostico
df_final_norm['diagnostic_superclass'] = df_final_norm['diagnostic_superclass'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)


df_final_norm = df_final_norm.dropna(subset=['diagnostic_superclass'])



df_balanceado_norm = df_final_norm.groupby('diagnostic_superclass', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 50), random_state=42)
)

print(df_balanceado_norm.iloc[0])
print(df_balanceado_norm.shape)

patient_id                                                          3423.0
age                                                                   88.0
sex                                                                      1
report                   sinus rhythm. left axis deviation left anterio...
infarction_stadium1                                                    NaN
filename_lr                                      records100/01000/01663_lr
filename_hr                                      records500/01000/01663_hr
diagnostic_superclass                                                   CD
IMC                                                              20.028842
Name: 1663, dtype: object
(200, 9)


/tmp/ipykernel_7862/3099351835.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanceado_norm = df_final_norm.groupby('diagnostic_superclass', group_keys=False).apply(


In [131]:
import wfdb
import matplotlib.pyplot as plt
import os

os.makedirs("temp_images_norm", exist_ok=True)

for ecg_id in df_balanceado_norm.index:
    # pega a linha do paciente no df principal (que tem o filename_lr)
    paciente = df.loc[ecg_id]
    nome_arquivo = f"temp_images_norm/{paciente['patient_id']}.png"

    # checa se já existe ANTES de processar, pra economizar tempo
    if os.path.exists(nome_arquivo):
        print(f"Já existe, pulando: {nome_arquivo}")
        continue

    record_path = os.path.join(base_path, paciente['filename_lr'])

    # lê o sinal
    record = wfdb.rdrecord(record_path)

    # plota e salva
    fig = wfdb.plot_wfdb(record=record, title=f'ECG - paciente {ecg_id}', return_fig=True)
    fig.savefig(nome_arquivo, dpi=150)
    plt.close(fig)  # fecha a figura pra não acumular memória no loop

    print(f"Salvo: {nome_arquivo}")

print("Concluído!")

Salvo: temp_images_norm/3423.0.png
Salvo: temp_images_norm/389.0.png
Salvo: temp_images_norm/3469.0.png
Salvo: temp_images_norm/3643.0.png
Salvo: temp_images_norm/6591.0.png
Salvo: temp_images_norm/2881.0.png
Salvo: temp_images_norm/7802.0.png
Salvo: temp_images_norm/4257.0.png
Salvo: temp_images_norm/928.0.png
Salvo: temp_images_norm/7106.0.png
Salvo: temp_images_norm/1914.0.png
Salvo: temp_images_norm/3640.0.png
Salvo: temp_images_norm/2103.0.png
Salvo: temp_images_norm/7149.0.png
Salvo: temp_images_norm/6128.0.png
Salvo: temp_images_norm/5067.0.png
Salvo: temp_images_norm/4428.0.png
Salvo: temp_images_norm/7640.0.png
Salvo: temp_images_norm/3157.0.png
Salvo: temp_images_norm/4064.0.png
Salvo: temp_images_norm/8340.0.png
Salvo: temp_images_norm/7002.0.png
Salvo: temp_images_norm/4902.0.png
Salvo: temp_images_norm/21541.0.png
Salvo: temp_images_norm/3870.0.png
Salvo: temp_images_norm/865.0.png
Salvo: temp_images_norm/901.0.png
Salvo: temp_images_norm/7497.0.png
Salvo: temp_images_norm

In [134]:
import os

pasta = "temp_images_norm"
qtd = len(os.listdir(pasta))
print(f"Total de arquivos: {qtd}")

Total de arquivos: 200


In [135]:
import shutil
from google.colab import files

# Compacta a pasta "temp_images_norm" em um arquivo zip
shutil.make_archive("temp_images_norm", 'zip', "temp_images_norm")

# Cria o arquivo zip
files.download("temp_images_norm.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Concatenando os dois datasets tratados (infarto , não infarto)

In [138]:
df_concat = pd.concat([df_balanceado_norm, df_balanceado_inf], ignore_index = True)

In [139]:
print(df_concat.iloc[0])
print(df_concat.shape)

patient_id                                                          3423.0
age                                                                   88.0
sex                                                                      1
report                   sinus rhythm. left axis deviation left anterio...
infarction_stadium1                                                    NaN
filename_lr                                      records100/01000/01663_lr
filename_hr                                      records500/01000/01663_hr
diagnostic_superclass                                                   CD
IMC                                                              20.028842
Name: 0, dtype: object
(250, 9)


In [142]:
df_type = df_concat['diagnostic_superclass'].unique()
print(df_type)

['CD' 'HYP' 'NORM' 'STTC' 'MI']



NORM-> Normal ECG ->	ECG normal, sem alterações

MI->	Myocardial Infarction->	Infarto do miocárdio (o que você quer)

STTC->	ST/T Change->	Alterações no segmento ST ou onda T (isquemia, repolarização)

CD	->Conduction Disturbance	->Distúrbio de condução (ex: bloqueios de ramo, BAV)

HYP	->Hypertrophy	->Hipertrofia (ex: do ventrículo esquerdo)

#Cria dados sinteticos para cada paciente , seguindo um padrão conforme sintomas e informações propicias ao diagnostico

In [143]:
import random

#pressão = <90- baixa / 90<120 - normal / 120<129 - normal alta / 130<139 - pre-hipertensão / 140<179 hipertensão / 180>= hipertensão emergencia
#colesterol_total = <200 - Bom / 200<239 - atenção / 240> alto
import random

def gerar_valores(row):
    if row['diagnostic_superclass'] == 'NORM':
        row["pressao"] = random.randint(90,140)
        row["diabetes"] = random.randint(0,1)
        row["doenca_cardiaca"] = random.randint(0,1)
        row["colesterol_total"] = random.randint(170,239)
        row["dor_peito"] = random.randint(0,1)
        row["dor_irradiada"] = random.randint(0,1)
        row["nausea"] = random.randint(0,1)
        row["falta_de_ar"] = random.randint(0,1)

    elif row['diagnostic_superclass'] == 'MI':
        row["pressao"] = random.randint(120,180)
        row["diabetes"] = random.randint(0,1)
        row["doenca_cardiaca"] = random.randint(0,1)
        row["colesterol_total"] = random.randint(170,250)
        row["dor_peito"] = random.choices([0,1], weights=[0.2,0.8], k=1)[0]
        row["dor_irradiada"] = random.choices([0,1], weights=[0.2,0.8], k=1)[0]
        row["nausea"] = random.choices([0,1], weights=[0.3,0.7], k=1)[0]
        row["falta_de_ar"] = random.choices([0,1], weights=[0.4,0.6], k=1)[0]

    elif row['diagnostic_superclass'] == 'STTC':
        row["pressao"] = random.randint(90,140)
        row["diabetes"] = random.randint(0,1)
        row["doenca_cardiaca"] = random.randint(0,1)
        row["colesterol_total"] = random.randint(170,239)
        row["dor_peito"] = random.randint(0,1)
        row["dor_irradiada"] = random.randint(0,1)
        row["nausea"] = random.randint(0,1)
        row["falta_de_ar"] = random.randint(0,1)

    elif row['diagnostic_superclass'] == 'CD':
        row["pressao"] = random.randint(90,140)
        row["diabetes"] = random.randint(0,1)
        row["doenca_cardiaca"] = random.randint(0,1)
        row["colesterol_total"] = random.randint(170,239)
        row["dor_peito"] = random.randint(0,1)
        row["dor_irradiada"] = random.randint(0,1)
        row["nausea"] = random.randint(0,1)
        row["falta_de_ar"] = random.randint(0,1)

    elif row['diagnostic_superclass'] == 'HYP':
        row["pressao"] = random.randint(90,140)
        row["diabetes"] = random.randint(0,1)
        row["doenca_cardiaca"] = random.randint(0,1)
        row["colesterol_total"] = random.randint(170,239)
        row["dor_peito"] = random.randint(0,1)
        row["dor_irradiada"] = random.randint(0,1)
        row["nausea"] = random.randint(0,1)
        row["falta_de_ar"] = random.randint(0,1)

    return row

df_dados = df_concat.apply(gerar_valores, axis=1)



In [170]:
from sklearn.preprocessing import LabelEncoder

#transformar os dados da coluna report em numerico para treinamento
le = LabelEncoder()
df_dados['report'] = le.fit_transform(df_dados['report'])

print(df_dados[['report', 'report']])

     report  report
0        68      68
1        71      71
2        14      14
3        64      64
4        62      62
..      ...     ...
245      56      56
246     186     186
247      85      85
248      17      17
249     141     141

[250 rows x 2 columns]


In [169]:
from sklearn.preprocessing import LabelEncoder

#transformar os dados da coluna infarction_stadium1  em numerico para treinamento
le = LabelEncoder()
df_dados['infarction_stadium1'] = le.fit_transform(df_dados['infarction_stadium1'])

print(df_dados[['infarction_stadium1', 'infarction_stadium1']])

     infarction_stadium1  infarction_stadium1
0                      4                    4
1                      4                    4
2                      4                    4
3                      4                    4
4                      4                    4
..                   ...                  ...
245                    2                    2
246                    3                    3
247                    2                    2
248                    2                    2
249                    3                    3

[250 rows x 2 columns]


In [173]:

df_dados['IMC'] = df_dados['IMC'].round(2)




In [174]:
print(df_dados.iloc[50])

patient_id                                   887.0
age                                           86.0
sex                                              1
report                                         119
infarction_stadium1                              4
filename_lr              records100/07000/07973_lr
filename_hr              records500/07000/07973_hr
diagnostic_superclass                          HYP
IMC                                          20.32
pressao                                        129
diabetes                                         0
doenca_cardiaca                                  1
colesterol_total                               191
dor_peito                                        0
dor_irradiada                                    0
nausea                                           0
falta_de_ar                                      0
Name: 50, dtype: object


In [177]:
df_dados.shape

(250, 17)

In [175]:
df_dados.to_csv('dataset_dados.csv', index=True)